# CropCare AI: training on Kaggle

Runs the whole AI pipeline on a free Kaggle GPU:

1. Prepare manifests for PlantVillage (lab), PlantDoc (field) and optional rice / wheat / maize datasets
2. **Experiment A**: train on PlantVillage only, then test on lab *and* field photos (shows the gap)
3. **Experiment B**: train on lab + field photos (the model the app uses)
4. Calibrate, evaluate, and print the comparison table
5. Train the severity (lesion segmentation) model on PlantSeg
6. Export ONNX and download everything as one zip

**Before running:** *Settings → Accelerator → GPU T4 x2 (or P100)*, *Settings → Internet → On*,
and *Add Data → search "plantvillage dataset" (abdallahalidev/plantvillage-dataset)*.
Optionally add rice / wheat / maize disease datasets too.

In [ ]:
REPO = "https://github.com/MohammadAftabAlam/Crop-Disease-Detection.git"
!git clone -q $REPO /kaggle/working/repo
%cd /kaggle/working/repo/ai-service
!pip install -q timm segmentation-models-pytorch pydantic-settings onnx onnxruntime
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU accelerator")

## 1. Find the datasets

In [ ]:
import glob, os

def find_dataset_root(marker_folder):
    """Return the folder that directly contains a class folder named `marker_folder`."""
    hits = glob.glob(f"/kaggle/input/**/{marker_folder}", recursive=True)
    hits = [h for h in hits if "segmented" not in h.lower() and "grayscale" not in h.lower()]
    return os.path.dirname(sorted(hits, key=len)[0]) if hits else None

PLANTVILLAGE = find_dataset_root("Tomato___Late_blight") or find_dataset_root("Tomato_Late_blight")
print("PlantVillage:", PLANTVILLAGE)

# PlantDoc (field photos) comes from GitHub
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /kaggle/working/PlantDoc-Dataset
PLANTDOC = "/kaggle/working/PlantDoc-Dataset"

# Optional crop datasets: set to the folder that contains one sub-folder per class, or leave None
RICE = None    # e.g. "/kaggle/input/<rice-dataset>/<folder with class folders>"
WHEAT = None
MAIZE = None

## 2. Manifests (lists of images with class and split; nothing is copied)

In [ ]:
!python -m cropcare_ai.data.prepare --name plantvillage --root "$PLANTVILLAGE"
# PlantDoc keeps its own train/test; 20% of its train becomes 'val', used to calibrate on field photos
!python -m cropcare_ai.data.prepare --name plantdoc --root "$PLANTDOC" --layout keep --val-from-train 0.2
for name, root in [("rice", RICE), ("wheat", WHEAT), ("maize", MAIZE)]:
    if root:
        !python -m cropcare_ai.data.prepare --name $name --root "$root" --crop $name

## 3. Experiment A: PlantVillage only (the usual student project)
Expect very high lab accuracy and a big drop on PlantDoc. That drop is the problem statement.

In [ ]:
!python -m cropcare_ai.training.train_classifier --config configs/train/a_plantvillage_only_mobilenetv3.yaml
A = "artifacts/exp_a_pv_only_mobilenetv3"
!python -m cropcare_ai.training.calibrate --model-dir $A --manifest data/manifests/plantdoc.csv --split val
!python -m cropcare_ai.training.evaluate --model-dir $A --manifest data/manifests/plantvillage.csv --name plantvillage_test --description "PlantVillage test (lab photos)"
!python -m cropcare_ai.training.evaluate --model-dir $A --manifest data/manifests/plantdoc.csv --name plantdoc_test --description "PlantDoc test (field photos)"

## 4. Experiment B: lab + field photos (the app's model)

In [ ]:
!python -m cropcare_ai.training.train_classifier --config configs/train/b_mixed_convnext_tiny.yaml
B = "artifacts/classifier"
!python -m cropcare_ai.training.calibrate --model-dir $B --manifest data/manifests/plantdoc.csv --split val
!python -m cropcare_ai.training.evaluate --model-dir $B --manifest data/manifests/plantvillage.csv --name plantvillage_test --description "PlantVillage test (lab photos)"
!python -m cropcare_ai.training.evaluate --model-dir $B --manifest data/manifests/plantdoc.csv --name plantdoc_test --description "PlantDoc test (field photos)"

### Optional: compare backbones (each ~same data as B)
Run any of these, then calibrate + evaluate them the same way with `--model-dir artifacts/<name>`.

In [ ]:
# !python -m cropcare_ai.training.train_classifier --config configs/train/b_mixed_dinov2_small.yaml
# !python -m cropcare_ai.training.train_classifier --config configs/train/b_mixed_efficientnetv2_b0.yaml
# !python -m cropcare_ai.training.train_classifier --config configs/train/b_mixed_mobilenetv3.yaml
# !python -m cropcare_ai.training.train_classifier --config configs/train/b_mixed_resnet50.yaml

## 5. The comparison table

In [ ]:
!python -m cropcare_ai.training.report --model-dir artifacts/exp_a_pv_only_mobilenetv3 artifacts/classifier

## 6. Severity model (PlantSeg, 1.7 GB, CC-BY 4.0)
The cell finds the COCO annotation files after unzipping; check the printed paths.

In [ ]:
!wget -q -O /kaggle/working/plantseg.zip "https://zenodo.org/records/14935094/files/plantsegv3.zip?download=1"
!unzip -q -o /kaggle/working/plantseg.zip -d /kaggle/working/plantseg && rm /kaggle/working/plantseg.zip
annotations = sorted(glob.glob("/kaggle/working/plantseg/**/*.json", recursive=True))
print("\n".join(annotations))

In [ ]:
import json
def image_folder_for(annotation_file):
    """Folder that the COCO file's image paths are relative to."""
    first = json.load(open(annotation_file))["images"][0]["file_name"]
    hits = glob.glob(f"/kaggle/working/plantseg/**/{os.path.basename(first)}", recursive=True)
    hit = next((h for h in hits if h.endswith(first)), None)
    return hit[: -len(first)].rstrip("/") if hit else None

TRAIN_JSON = next(a for a in annotations if "train" in os.path.basename(a).lower())
VAL_JSON = next(a for a in annotations if "val" in os.path.basename(a).lower())
TRAIN_IMAGES, VAL_IMAGES = image_folder_for(TRAIN_JSON), image_folder_for(VAL_JSON)
print(TRAIN_JSON, TRAIN_IMAGES, VAL_JSON, VAL_IMAGES, sep="\n")

In [ ]:
!python -m cropcare_ai.training.train_segmenter --train-coco "$TRAIN_JSON" --train-images "$TRAIN_IMAGES" --val-coco "$VAL_JSON" --val-images "$VAL_IMAGES" --epochs 25

## 7. Export and download

In [ ]:
!python -m cropcare_ai.training.export_onnx --model-dir artifacts/classifier
!cp data/manifests/*.json artifacts/ 2>/dev/null; cd artifacts && zip -qr /kaggle/working/cropcare_artifacts.zip . -x "*.onnx"
!ls -lh /kaggle/working/cropcare_artifacts.zip
print("Download cropcare_artifacts.zip from the Output panel and unzip it into ai-service/artifacts/ on your laptop.")